# Optimising a Simple Stock Investment Portfolio

**Note:** These examples are inspired by James, Gareth, Daniela Witten, Trevor Hastie, and Rob Tibshirani. An introduction to statistical learning. Springer, 2013.

In [4]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

## Introduction

### Scenario:

- There are two correlated stocks, $X$ and $Y$, both of which are normally distributed.

- All available money is invested in these two stocks.

- The goal is to find $\alpha \in [0,1]$ (the fraction of your investment allocated to stock $X$) that minimises the variance of the portfolio, i.e.: $$ \mathbb{V}\mathrm{AR}[\alpha X + (1-\alpha)Y] $$

- `returns.npy` = two-dimensional NumPy array that contains the past $200$ return values for $(X,Y)$.

### The aim of the notebook is to:

1. Estimate $\mu_X, \mu_Y, \sigma_{X}^2, \sigma_{Y}^2, \sigma_{XY}$ from the sample and compute the optimal investment strategy $\alpha$.

2. Since the parameters were estimated from a sample, the results may be biased. Therefore...
- Use bootstrapping, repeatedly sampling 200 return values to estimate the optimal investment strategy each time.
- The sampling is repeated $B = 500$ times, and the standard error of the optimal $\alpha$ estimated from the original return data.

###1) Load the Dataset

In [9]:
returns = np.load('/content/returns.npy')
print(returns[:5])
n = 200
print("The shape of returns is ",returns.shape)

[[3.68186601 3.44400631]
 [3.36577606 2.63885614]
 [4.52691953 4.56078022]
 [1.48572422 2.87869915]
 [4.75081789 0.74025049]]
The shape of returns is  (200, 2)


###2) Estimate the mean returns (expected return)

In [10]:
hat_mean = np.mean(returns, 0)
print("The mean returns is given by ",hat_mean)

The mean returns is given by  [2.97259589 2.98935547]


### 3) Compute the covariance matrix.
This is how much the two stocks relate to each other.\
Here is the formula for the covariance matrix:

$$
\text{Cov}(X, Y) = \frac{1}{n - 1} \sum_{i=1}^{n} (x_i - \mu_X)(y_i - \mu_Y)
$$

In [11]:
hat_cov = np.zeros((2,2))
for i in range(n):
    hat_cov = hat_cov + (returns[i]-hat_mean).reshape((2,1)) * (returns[i]-hat_mean).reshape((1,2))

hat_cov = hat_cov/(n-1)
hat_varx = hat_cov[0,0]
hat_vary = hat_cov[1,1]
hat_covar = hat_cov[0,1]

print("The covariance matrix is given by ",hat_cov)

The covariance matrix is given by  [[1.29126685 0.46509466]
 [0.46509466 1.10756609]]


##Estimate the optimal investment

The optimal investment is as follows:

$$ \alpha = \frac{\sigma_Y^2 - \text{Cov}(X,Y)}{\sigma_X^2 + \sigma_Y^2 - 2 \text{Cov}(X,Y)} $$

where:

- $\sigma_X^2$ is the variance of asset X.

- $\sigma_Y^2$ is the variance of asset Y.

- $\text{Cov}(X,Y)$ is the covariance between X and Y.

Estimate the optimal value of $\alpha$.

In [12]:
def optimal_alpha(varx, vary, covar):
    alpha = (vary - covar) / (varx + vary - 2*covar)
    return alpha

optimal_investment = optimal_alpha(hat_varx, hat_vary, hat_covar)

print("The optimal investment is given by ",round(optimal_investment,3) )

The optimal investment is given by  0.437


In [13]:
def hat_alpha(sample_returns): #make the above process a function
    hat_mean = np.mean(sample_returns,0)
    hat_cov = np.zeros((2,2))
    n = np.size(sample_returns,0)
    for i in range(n):
        hat_cov = hat_cov + (sample_returns[i]-hat_mean).reshape((2,1))*(sample_returns[i]-hat_mean).reshape((1,2))
    hat_cov = hat_cov/(n-1)
    hat_varx = hat_cov[0,0]
    hat_vary = hat_cov[1,1]
    hat_covar = hat_cov[0,1]
    return optimal_alpha(hat_varx, hat_vary, hat_covar)

## Using Bootstrapping to Obtain an Approximation for $\hat{\alpha}$


In [15]:
simulation = 500
B = simulation
samples = 200
estimations = np.zeros(simulation)

for sim in range(simulation): #Simulate this many times
    generated_sample = returns[np.random.choice(returns.shape[0], size=samples,replace=True), :]
    estimations[sim] = hat_alpha(generated_sample)


print("mean", round(np.mean(estimations),3),"min",np.min(estimations), "max", np.max(estimations))

mean 0.442 min 0.2559073459468998 max 0.5944826298126019


Given a set of bootstrap estimates $(\hat{\alpha}_1, \hat{\alpha}_2, \dots, \hat{\alpha}_B)$, the standard error of the mean (SEM) is calculated as:

$$
\text{SEM} = \sqrt{\frac{1}{B-1} \sum_{i=1}^{B} (\hat{\alpha}_i - \bar{\hat{\alpha}})^2}
$$

where:

-  B is the number of bootstrap simulations.
- $(\bar{\hat{\alpha}} = \frac{1}{B} \sum_{i=1}^{B} \hat{\alpha}_i)$ is the mean of the bootstrap estimates.
- $(\hat{\alpha}_i)$ is the estimate from the \( i \)-th bootstrap simulation.


####Computing the MSE across the bootstrap estimations of $\hat{\alpha}$

In [16]:
bar_alpha = np.mean(estimations)
bootstrap_error = np.sqrt(np.sum(((estimations - bar_alpha)**2))/ (B-1))

print("The bootstrap error is given by ",round(bootstrap_error,3))

The bootstrap error is given by  0.052


###Computing the MSE for bootstrap estimations with sample = 100

In [ ]:
B = 500
samples = 100
estimations = np.zeros(simulation)

for sim in range(simulation):
    sample = returns[np.random.choice(returns.shape[0], size = samples, replace = True), :]
    estimations[sim] = hat_alpha(sample)

bar_alpha = np.mean(estimations)
bootstrap_error1 = np.sqrt(np.sum(((estimations - bar_alpha)**2))/ (B-1))

print("The bootstrap error is given by ",round(bootstrap_error1,3))

The bootstrap error is given by  0.069
